# Edge Computing Assignment 4 — KD + Hardware-aware NAS (Colab runner)
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Quick mode takes about 2–2.5 h in total.

In [ ]:
# 1) get the code (upload the repo zip to Colab, or clone your GitHub repo)
# !git clone https://github.com/<your-user>/<your-repo>.git edge-kd-nas
# or: from google.colab import files; files.upload()  then  !unzip -q edge-kd-nas.zip
%cd edge-kd-nas
!pip -q install onnx onnxruntime optuna psutil

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Q1 — teacher, baselines, distillation

In [ ]:
!python -m q1_kd.train_teacher --pretrained --epochs 30 --lr 0.05 --amp

In [ ]:
!python -m q1_kd.train_baseline --arch all --epochs 30 --amp

In [ ]:
!python -m q1_kd.train_kd --arch all --epochs 30 --teacher checkpoints/teacher_resnet50.pt --T 4 --alpha 0.9 --beta 1.0 --amp

## Q2 — hardware-aware NAS on CIFAR-10

In [ ]:
!python -m q2_nas.search --strategy evolution --budget 40 --population 10 --proxy-epochs 5 --max-latency-ms 8 --max-size-mb 2 --max-peak-mem-mb 64 --amp

In [ ]:
!python -m q2_nas.train_best --epochs 60 --amp

## Deployment, benchmarking, report figures

In [ ]:
!python -m deploy.export_onnx --include-teacher
!python -m benchmark.benchmark --threads 1 --runs 100
!python -m deploy.infer --all --cifar-samples 5
!python -m deploy.onnx_infer --all --cifar-samples 5
!python -m tools.make_report_assets

In [ ]:
# 4) download everything (checkpoints, onnx, results) to put in the GitHub repo
!zip -qr outputs.zip checkpoints onnx results
from google.colab import files; files.download('outputs.zip')